# Two-stage XCH4: observational usage example

This Notebook demonstrates Stage2 bias correction using a prepared monthly collocation table. Stage1 usage is described at the end. Prepare the inputs listed in `DATA_README.md`; this code-only distribution does not bundle observational data.

The example uses the13-feature,800-tree `archived_loyo` preset. See `METHODS.md` for the core workflow and parameter choices.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import xarray as xr
from IPython.display import display, Image
from two_stage import run_example, plot_saved

source = Path('data/stage2_collocations_sample.csv')  # Set your prepared input path here.
initial_sha256 = hashlib.sha256(source.read_bytes()).hexdigest()
data = pd.read_csv(source)
display({'input_records': len(data), 'months': pd.to_datetime(data.time).dt.to_period('M').nunique()})
display(data.isna().sum().rename('missing_count').to_frame())

## Fit Stage2 and save predictions

Fit on2019–2022 rows and predict2023. Scaling is fitted on training data only. Missing numeric predictors remain NaN. Baseline and corrected scores use the same eligible rows. This illustrates conditional Stage2 evaluation, not all experiments in the manuscript.

In [ ]:
scores = run_example(data_path=source, preset='archived_loyo')
print(json.dumps(scores, indent=2))
display(pd.DataFrame(scores).T)
predictions = pd.read_csv('outputs/holdout_predictions.csv')
assert len(predictions) == int(pd.to_datetime(data.time).dt.year.eq(2023).sum())
assert scores['baseline']['n'] == scores['corrected']['n']
display(predictions[['row_id', 'time', 'XCH4_go', 'XCH4_pred', 'predicted_delta', 'XCH4_bc']].head())

## Plot saved results

Plotting reads the saved CSV and does not train a model. Both panels use exactly the same eligible records. Correction need not improve every record or every holdout.

In [ ]:
fig = plot_saved()
display(Image(filename='outputs/holdout_scatter.png'))

## Inspect sparse NetCDF

The `sample` dimension stores individual collocation records with per-record time/latitude/longitude and source CSV row IDs. It is **not** a complete national 0.1° Cartesian grid. Concentrations and residuals are in ppb.

In [ ]:
with xr.open_dataset('outputs/holdout_predictions.nc') as ds:
    assert dict(ds.sizes) == {'sample': len(predictions)}
    display({'dimensions': dict(ds.sizes), 'coverage': ds.attrs['coverage'],
             'XCH4_bc_units': ds.XCH4_bc.attrs['units']})
assert hashlib.sha256(source.read_bytes()).hexdigest() == initial_sha256
display('Source CSV unchanged; CSV, NetCDF, model and PNG saved in outputs/.')

## Stage1 usage

Prepare a training table with the environmental predictors, valid coordinates/months and observed TROPOMI `XCH4_tr`. Mapping and GOSAT values are not replacements for that target.

```python
from two_stage import tune_stage1, fit_stage1
parameters = tune_stage1(stage1_training_table)
mapping_model = fit_stage1(stage1_training_table, params=parameters, base_year=2019)
mapping_predictions = mapping_model.predict(full_feature_table)
```

`cross_validate` supplies sample, spatial and month-group checks for prepared inputs. `fit_stage2(frame, preset="primary")` selects the early11-feature branch; `preset="archived_loyo"` selects the13-feature branch used above.